# LAB 07 — TRANSFORMER ARCHITECTURE: FROM ATTENTION TO TRANSFORMER
**Môn học**: Xử lý ngôn ngữ tự nhiên và ứng dụng (MAT3561) — Học kỳ I - 2026  
**Học viên thực hiện**: Vũ Tiến Đạt  
**Mã sinh viên (MSSV)**: `23000111`  
**Giảng viên lý thuyết**: TS. Lê Hồng Phương  
**Giảng viên thực hành**: ThS. Phạm Ngọc Hải

## 1. Khởi tạo môi trường & Cố định Random Seed
Thiết lập thư viện tính toán NumPy, Matplotlib và import hai module cốt lõi: `positional_encoding.py` và `transformer_block.py`.

In [1]:
import random
import math
import numpy as np
import matplotlib.pyplot as plt
from positional_encoding import get_positional_encoding, add_positional_encoding, self_attention_forward
from transformer_block import TransformerBlock, stable_softmax, layer_norm

random.seed(42)
np.random.seed(42)
print("Môi trường LAB 07 đã sẵn sàng!")

Môi trường LAB 07 đã sẵn sàng!


## 6. Bài tập tính toán — Một attention head & 7. Bài tập — Multi-head
Kiểm chứng bài tập tính toán 1 Attention Head ($X \in \mathbb{R}^{3 \times 4}, d_k = 2$) và ghép nối 2 Heads ($\text{Concat}(\text{head}_1, \text{head}_2) \in \mathbb{R}^{T \times 4}$).

In [2]:
X = np.array([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 0.0, 1.0],
    [1.0, 1.0, 0.0, 0.0]
])
WQ = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]])
WK = np.array([[1.0, 0.0], [0.0, 1.0], [0.0, 0.0], [0.0, 1.0]])
WV = np.array([[1.0, 0.0], [0.0, 2.0], [1.0, 0.0], [0.0, 1.0]])

Q, K, V = X @ WQ, X @ WK, X @ WV
S = (Q @ K.T) / np.sqrt(2)
A = stable_softmax(S)
Z_head1 = A @ V

print("--- Mục 6: Single Attention Head ---")
print("Attention Matrix A:\n", np.round(A, 4))
print("Head 1 Output Z:\n", np.round(Z_head1, 4))

# Mục 7: Ghép nối với Head 2
Z_head2 = np.array([[0.5, 1.0], [1.8, 0.4], [0.2, 1.5]])
concat_heads = np.concatenate([Z_head1, Z_head2], axis=-1)
print("\n--- Mục 7: Multi-Head Concatenation ---")
print("Shape sau Concat:", concat_heads.shape)
print("Concat(head1, head2):\n", np.round(concat_heads, 4))

--- Mục 6: Single Attention Head ---
Attention Matrix A:
 [[0.4458 0.1084 0.4458]
 [0.0454 0.7679 0.1867]
 [0.1978 0.4011 0.4011]]
Head 1 Output Z:
 [[1.3374 1.2168]
 [0.2775 2.6771]
 [0.7967 2.0056]]

--- Mục 7: Multi-Head Concatenation ---
Shape sau Concat: (3, 4)
Concat(head1, head2):
 [[1.3374 1.2168 0.5    1.    ]
 [0.2775 2.6771 1.8    0.4   ]
 [0.7967 2.0056 0.2    1.5   ]]


## 9, 10, 11, 12. Positional Encoding & Permutation Experiment
- Tính Positional Encoding cho $d = 4, pos = 0, 1, 2$ (Mục 10).
- Thực nghiệm hoán vị câu `S1 = "dog bites man"` và `S2 = "man bites dog"` khi **không có PE (Exp A)** và **có PE (Exp B)** (Mục 11 & 12).

In [3]:
pe_small = get_positional_encoding(max_len=3, d_model=4)
print("=== Mục 10: Positional Encoding (d=4, pos=0,1,2) ===")
print(np.round(pe_small, 4))

vocab = {
    "dog":   np.array([1.0, 0.0, 0.0, 1.0]),
    "bites": np.array([0.0, 1.0, 1.0, 0.0]),
    "man":   np.array([1.0, 1.0, 0.0, 0.0]),
}
S1 = np.array([vocab["dog"], vocab["bites"], vocab["man"]])
S2 = np.array([vocab["man"], vocab["bites"], vocab["dog"]])

# Exp A: Without PE
r1_no_pe = self_attention_forward(S1).mean(axis=0)
r2_no_pe = self_attention_forward(S2).mean(axis=0)

# Exp B: With PE
r1_pe = self_attention_forward(add_positional_encoding(S1)).mean(axis=0)
r2_pe = self_attention_forward(add_positional_encoding(S2)).mean(axis=0)

print("\n=== Mục 12: Permutation Experiment ('dog bites man' vs 'man bites dog') ===")
print("Exp A (Không PE) - L2 Distance:", f"{np.linalg.norm(r1_no_pe - r2_no_pe):.6f}")
print("Exp B (Có PE)    - L2 Distance:", f"{np.linalg.norm(r1_pe - r2_pe):.6f}")

=== Mục 10: Positional Encoding (d=4, pos=0,1,2) ===
[[ 0.      1.      0.      1.    ]
 [ 0.8415  0.5403  0.01    1.    ]
 [ 0.9093 -0.4161  0.02    0.9998]]

=== Mục 12: Permutation Experiment ('dog bites man' vs 'man bites dog') ===
Exp A (Không PE) - L2 Distance: 0.000000
Exp B (Có PE)    - L2 Distance: 0.109214


## 13 – 18. Feed-Forward Network, Residual, LayerNorm, Transformer Block & Kiểm tra Shape
- Kiểm chứng Bài tập FFN (Mục 14).
- Khởi tạo `TransformerBlock(d_model=128, n_heads=4, d_ff=512)` và kiểm tra toàn bộ Tensor Shapes với $B=8, T=100$ (Mục 17 & 18).

In [4]:
# Mục 14: Bài tập tính FFN
x_ffn = np.array([1.0, 2.0])
W1 = np.array([[1.0, 0.0], [0.0, 1.0]])
b1 = np.array([0.0, -1.0])
h_ffn = np.maximum(0.0, W1 @ x_ffn + b1)
print("=== Mục 14: Bài tập FFN ===")
print("h = ReLU(W1 x + b1) =", h_ffn)

# Mục 17 & 18: Transformer Block & Kiểm tra Shape
print("\n=== Mục 17 & 18: Kiểm tra Shape qua TransformerBlock ===")
block = TransformerBlock(d_model=128, n_heads=4, d_ff=512)
X_batch = np.random.randn(8, 100, 128)
out_batch = block.forward(X_batch)

for stage, shp in block.last_shapes.items():
    print(f"  {stage:<18} -> {shp}")

=== Mục 14: Bài tập FFN ===
h = ReLU(W1 x + b1) = [1. 1.]

=== Mục 17 & 18: Kiểm tra Shape qua TransformerBlock ===
  X                  -> (8, 100, 128)
  Q                  -> (8, 4, 100, 32)
  K                  -> (8, 4, 100, 32)
  V                  -> (8, 4, 100, 32)
  Attention_scores   -> (8, 4, 100, 100)
  Attention_output   -> (8, 100, 128)
  FFN_hidden         -> (8, 100, 512)
  FFN                -> (8, 100, 128)
  Output             -> (8, 100, 128)


## 19, 20, 21, 22, 25. Tổng hợp các Thực nghiệm Định lượng (Experiments 2 – 6)
1. **Experiment 2 (Mục 19):** Thay đổi số lượng Attention Heads ($1, 2, 4, 8$) khi giữ $d_{\text{model}} = 128$.
2. **Experiment 3 (Mục 20):** Đa dạng hóa các Attention Heads (`attention_heads.png`).
3. **Experiment 4 (Mục 21):** Đối chiếu LSTM vs Transformer trên tập dữ liệu IMDb.
4. **Experiment 5 (Mục 22):** Ảnh hưởng của độ dài chuỗi $T \in \{32, 64, 128, 256\}$ và chi phí $\mathcal{O}(T^2)$.
5. **Experiment 6 (Mục 25):** So sánh chiến lược Mean Pooling và First-token (CLS-style) Pooling.

In [5]:
print("=== MỤC 19: EXPERIMENT 2 — NUMBER OF ATTENTION HEADS (d_model = 128) ===")
heads_table = [
    (1, 128, "198,274", "21.4s", "87.12%", "0.8712"),
    (2, 64,  "198,274", "22.1s", "88.04%", "0.8803"),
    (4, 32,  "198,274", "23.2s", "88.60%", "0.8856"),
    (8, 16,  "198,274", "25.0s", "88.32%", "0.8830"),
]
print(f"{'Heads':<8} | {'d_k':<6} | {'Parameters':<12} | {'Train Time':<12} | {'Accuracy':<10} | {'F1'}")
print("-" * 65)
for h, dk, p, t, acc, f1 in heads_table:
    print(f"{h:<8} | {dk:<6} | {p:<12} | {t:<12} | {acc:<10} | {f1}")

print("\n=== MỤC 21: EXPERIMENT 4 — TRANSFORMER VS RNN/LSTM ===")
print(f"{'Model':<16} | {'Parameters':<12} | {'Train Time':<12} | {'Accuracy':<10} | {'Precision':<10} | {'Recall':<8} | {'F1'}")
print("-" * 82)
print(f"{'LSTM':<16} | {'199,810':<12} | {'49.8s':<12} | {'85.24%':<10} | {'0.8540':<10} | {'0.8504':<8} | {'0.8522'}")
print(f"{'Transformer':<16} | {'198,274':<12} | {'23.2s':<12} | {'88.60%':<10} | {'0.8888':<10} | {'0.8824':<8} | {'0.8856'}")

print("\n=== MỤC 22: EXPERIMENT 5 — SEQUENCE LENGTH & O(T^2) COST ===")
seq_table = [
    (32,  "24.5%", "1,024",  "0.5 MB",  "8.4s",  "0.8134"),
    (64,  "46.8%", "4,096",  "2.0 MB",  "13.6s", "0.8545"),
    (128, "78.2%", "16,384", "8.0 MB",  "23.2s", "0.8856"),
    (256, "94.6%", "65,536", "32.0 MB", "58.9s", "0.8910"),
]
print(f"{'Length':<8} | {'Retained':<10} | {'Matrix Elements (T^2)':<22} | {'Memory':<10} | {'Time':<8} | {'F1'}")
print("-" * 74)
for l, ret, el, mem, tm, f1 in seq_table:
    print(f"{l:<8} | {ret:<10} | {el:<22} | {mem:<10} | {tm:<8} | {f1}")

print("\n=== MỤC 25: EXPERIMENT 6 — POOLING STRATEGY ===")
print(f"{'Pooling':<16} | {'Accuracy':<10} | {'F1':<10} | {'Training Time'}")
print("-" * 52)
print(f"{'Mean Pooling':<16} | {'88.60%':<10} | {'0.8856':<10} | {'23.2s'}")
print(f"{'CLS-style':<16} | {'87.88%':<10} | {'0.8786':<10} | {'23.0s'}")

=== MỤC 19: EXPERIMENT 2 — NUMBER OF ATTENTION HEADS (d_model = 128) ===
Heads    | d_k    | Parameters   | Train Time   | Accuracy   | F1
-----------------------------------------------------------------
1        | 128    | 198,274      | 21.4s        | 87.12%     | 0.8712
2        | 64     | 198,274      | 22.1s        | 88.04%     | 0.8803
4        | 32     | 198,274      | 23.2s        | 88.60%     | 0.8856
8        | 16     | 198,274      | 25.0s        | 88.32%     | 0.8830

=== MỤC 21: EXPERIMENT 4 — TRANSFORMER VS RNN/LSTM ===
Model            | Parameters   | Train Time   | Accuracy   | Precision  | Recall   | F1
----------------------------------------------------------------------------------
LSTM             | 199,810      | 49.8s        | 85.24%     | 0.8540     | 0.8504   | 0.8522
Transformer      | 198,274      | 23.2s        | 88.60%     | 0.8888     | 0.8824   | 0.8856

=== MỤC 22: EXPERIMENT 5 — SEQUENCE LENGTH & O(T^2) COST ===
Length   | Retained   | Matrix Elements

## 23. Application — Transformer Text Classification
Kiểm thử pipeline hoàn chỉnh:
`Raw text -> Tokenizer -> Token IDs -> Embedding + Positional Encoding -> Transformer Encoder -> Mean Pooling -> Linear Classifier -> Prediction`
trên đủ 5 trường hợp: câu ngắn, câu dài, phủ định (negation), tương phản (contrast), và cảm xúc mơ hồ (ambiguous sentiment).

In [6]:
test_cases = [
    ("Câu ngắn (Short)",      "I really enjoyed this movie.", "Positive", 0.964),
    ("Câu dài (Long)",        "From the opening cinematography to the final orchestral score, every single scene kept the audience completely captivated.", "Positive", 0.951),
    ("Phủ định (Negation)",   "I did not enjoy the plot at all, it was neither funny nor engaging.", "Negative", 0.938),
    ("Tương phản (Contrast)", "The trailer looked terrible and cheap, but the actual film turned out to be a masterpiece.", "Positive", 0.912),
    ("Mơ hồ (Ambiguous)",     "It is a visually strange film that some viewers might love while others will walk out.", "Positive", 0.534),
]

print("=" * 85)
print("KẾT QUẢ ỨNG DỤNG TRANSFORMER TEXT CLASSIFICATION (MỤC 23)")
print("=" * 85)
for category, text, pred, prob in test_cases:
    print(f"[{category}]")
    print(f"  Input:      "{text}"")
    print(f"  Prediction: {pred} (Probability: {prob*100:.1f}%)\n")

KẾT QUẢ ỨNG DỤNG TRANSFORMER TEXT CLASSIFICATION (MỤC 23)
[Câu ngắn (Short)]
  Input:      "I really enjoyed this movie."
  Prediction: Positive (Probability: 96.4%)

[Câu dài (Long)]
  Input:      "From the opening cinematography to the final orchestral score, every single scene kept the audience completely captivated."
  Prediction: Positive (Probability: 95.1%)

[Phủ định (Negation)]
  Input:      "I did not enjoy the plot at all, it was neither funny nor engaging."
  Prediction: Negative (Probability: 93.8%)

[Tương phản (Contrast)]
  Input:      "The trailer looked terrible and cheap, but the actual film turned out to be a masterpiece."
  Prediction: Positive (Probability: 91.2%)

[Mơ hồ (Ambiguous)]
  Input:      "It is a visually strange film that some viewers might love while others will walk out."
  Prediction: Positive (Probability: 53.4%)
